In [1]:
import time

import numpy as np
import pandas as pd
from hssm import HSSM

from drift_diffusion.model import DriftDiffusionModel
from drift_diffusion.sim import sample_from_pdf

In [2]:
# true parameters and sample sizes to test
true_params = {"a": 0.63, "t0": 0.435, "v": 2.23, "z": 0.008}
true_unc = pd.read_csv("./results/sim-constant-params.csv").std().round(3).to_dict()
param_map = {"a": "a", "t": "t0", "v": "v", "z": "z"}  # hssm -> ddm
sample_sizes = [100, 500, 1000, 10000]
results = []

In [3]:
REFIT = False  # True to refit model comparison, False to load precomputed results


def fit_mle(X, y):
    """MLE with non-robust uncertainties"""
    mle = DriftDiffusionModel(cov_estimator="sample-hessian", p_outlier=1e-12)
    t0 = time.time()
    mle.fit(X, y, params0=np.array([1.0, 0.1, 0.1, 0.1]))
    return time.time() - t0, mle.params_, np.sqrt(np.diag(mle.covariance_))


def fit_mcmc(y_df):
    """MCMC with non-robust uncertainties"""
    mcmc = HSSM(data=y_df, model="ddm")
    t0 = time.time()
    mcmc.sample(cores=1, quiet=True, initvals={"a": 1.0, "t": 0.1, "v": 0.1, "z": 0.55})
    runtime = time.time() - t0
    est, unc = mcmc.summary().loc[list(param_map), ["mean", "sd"]].to_numpy().T
    est[-1], unc[-1] = 2 * est[-1] - 1, 2 * unc[-1]  # rescale z to (-1, 1)
    return runtime, est, unc


# compare both models across sample sizes using aggregate MAE metrics

if REFIT:
    for n in sample_sizes:
        X = pd.DataFrame({"intercept": np.ones(n)})
        y = sample_from_pdf(**true_params, n_samples=n, random_state=n)
        y_df = pd.DataFrame({"rt": np.abs(y), "response": np.sign(y)})

        mle_runtime, mle_params_, mle_unc_ = fit_mle(X, y)
        mcmc_runtime, mcmc_params_, mcmc_unc_ = fit_mcmc(y_df)

        params = np.array([true_params[name] for name in param_map.values()])
        uncs = np.array([true_unc[name] for name in param_map.values()])

        mle_mae = np.mean(np.abs(mle_params_ - params))
        mcmc_mae = np.mean(np.abs(mcmc_params_ - params))
        mle_unc_mae = np.mean(np.abs(mle_unc_ - uncs))
        mcmc_unc_mae = np.mean(np.abs(mcmc_unc_ - uncs))

        results.append(
            {
                "n trials": n,
                "mle runtime (s)": mle_runtime,
                "mcmc runtime (s)": mcmc_runtime,
                "mcmc/mle runtime": mcmc_runtime / mle_runtime,
                "mle param mae": mle_mae,
                "mcmc param mae": mcmc_mae,
                "mle unc mae": mle_unc_mae,
                "mcmc unc mae": mcmc_unc_mae,
            }
        )
    tab01 = pd.DataFrame(results).round(3)
else:
    tab01 = pd.read_csv("./results/tab-mle-vs-mcmc.csv")

In [4]:
tab01

,n trials,mle runtime (s),mcmc runtime (s),mcmc/mle runtime,mle param mae,mcmc param mae,mle unc mae,mcmc unc mae
0,100,0.531,10.263,19.321,0.017,0.021,0.075,0.078
1,500,0.809,26.193,32.396,0.020,0.038,0.017,0.019
2,1000,0.683,47.140,69.050,0.046,0.029,0.002,0.001
3,10000,5.490,484.717,88.289,0.009,0.011,0.025,0.025
